# LangChain Document Splitting (Chunking) – Simple Practical Notebook

This notebook follows the simple style of your uploaded `Langchain_Chunking.ipynb`:
- Small explanations before each topic
- Individual import cells
- Separate code execution cells
- Simple examples with visible outputs
- No complicated combined code

## Topics

1. What is Chunking?
2. Why is Chunking Required?
3. CharacterTextSplitter
4. RecursiveCharacterTextSplitter
5. TokenTextSplitter
6. MarkdownHeaderTextSplitter
7. HTML Header Text Splitter
8. Chunk Size
9. Chunk Overlap
10. Semantic Chunking
11. Parent-Child Chunking

The `Data_files` folder contains a TXT file, Markdown file, and HTML file for practice.

## 0. Install Required Packages

Run these installation cells one by one.

- `langchain-text-splitters` – text splitting classes
- `tiktoken` – token counting and token-based splitting
- `langchain-experimental` – SemanticChunker
- `langchain-huggingface` and `sentence-transformers` – local embedding model for semantic chunking
- `beautifulsoup4` – HTML parsing support

In [1]:
%pip install -U langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install -U tiktoken

In [2]:
%pip install -U langchain-experimental

  Using cached langchain_experimental-0.4.2-py3-none-any.whl.metadata (1.6 kB)
Using cached langchain_experimental-0.4.2-py3-none-any.whl (211 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -U langchain-huggingface sentence-transformers

^C
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -U beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


## 1. What is Chunking?

Chunking means dividing a large document into smaller pieces.

Example:
- You have a 100-page PDF.
- Instead of sending all pages to the language model at once, you divide the text into smaller chunks.
- Each small piece is called a **chunk**.

A chunk is simply a smaller part of the original text.

### Create some sample text

In [5]:
text = """
LangChain is a framework for building applications with language models.
It provides tools that help developers connect language models with data.

Document loading is the first step in many RAG applications.
A loader reads files such as PDF, TXT, CSV, JSON, and Word documents.

Chunking divides long text into smaller pieces.
Smaller chunks can help a retriever find useful information.
"""

print(text)


LangChain is a framework for building applications with language models.
It provides tools that help developers connect language models with data.

Document loading is the first step in many RAG applications.
A loader reads files such as PDF, TXT, CSV, JSON, and Word documents.

Chunking divides long text into smaller pieces.
Smaller chunks can help a retriever find useful information.



### Check the type and length of the text

In [6]:
type(text)

str

In [7]:
len(text)

390

Here, `len(text)` counts characters, not words and not tokens.

In [8]:
print("Characters:", len(text))
print("Words:", len(text.split()))

Characters: 390
Words: 60


## 2. Why is Chunking Required?

### Simple explanation

1. Language models have a context window limit.
2. Very large documents may be difficult to process in one request.
3. Retrieval systems work with smaller pieces of text.
4. Smaller chunks can help return only the relevant information.
5. Overlap can help preserve context between neighboring chunks.

### Example without chunking

In [9]:
print(text)


LangChain is a framework for building applications with language models.
It provides tools that help developers connect language models with data.

Document loading is the first step in many RAG applications.
A loader reads files such as PDF, TXT, CSV, JSON, and Word documents.

Chunking divides long text into smaller pieces.
Smaller chunks can help a retriever find useful information.



### Example: Imagine splitting the text into small pieces
We will use a splitter in the next topic to do this automatically.

In [10]:
print("Total characters in original text:", len(text))

Total characters in original text: 390


## 3. CharacterTextSplitter

### What is CharacterTextSplitter?

`CharacterTextSplitter` divides text using one separator.

For example:
- `\n\n` – split using paragraphs
- `\n` – split using new lines
- `" "` – split using spaces
- `""` – split character by character

**Important:** CharacterTextSplitter first tries the separator. A resulting chunk can be larger than `chunk_size` if one indivisible piece is too large.

### Step 1: Import CharacterTextSplitter

In [11]:
from langchain_text_splitters import CharacterTextSplitter

  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   ---------------------------------------- 846.4/846.4 kB 5.6 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ------------------------------------- -- 3.7/3.9 MB 16.3 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 11.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   -------------- ------------------------- 1.0/2.9 MB 5.3 MB/s eta 0:00:01
   -------------------------------- ------- 2.4/2.9 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 5.2 MB/s  0:00:00
   ---------------------------------------- 0.0/780.3 kB ? eta -:--:--
   ---------------------------------------- 780.3/780.3 kB 7.2 MB/s  0:00:00
   ---------------------------------------- 0.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


### Step 2: Create a small text example

In [88]:
text = """Python is easy to learn.
Python is used in automation.

LangChain is used to build LLM applications.
Chunking divides long text into smaller pieces."""

print(text)

Python is easy to learn.
Python is used in automation.

LangChain is used to build LLM applications.
Chunking divides long text into smaller pieces.


### Step 3: Create the splitter
Here, `separator='\n\n'` means split at paragraph boundaries.

In [89]:
splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=50,
    chunk_overlap=0
)

### Step 4: Split the text

In [90]:
chunks = splitter.split_text(text)

Created a chunk of size 54, which is longer than the specified 50


### Step 5: Check how many chunks were created

In [91]:
len(chunks)

2

### Step 6: Print each chunk

In [92]:
for chunk in chunks:
    print(chunk)
    print("---------------")

Python is easy to learn.
Python is used in automation.
---------------
LangChain is used to build LLM applications.
Chunking divides long text into smaller pieces.
---------------


### Try splitting using a new line

In [93]:
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=50,
    chunk_overlap=0
)

chunks = splitter.split_text(text)
chunks

['Python is easy to learn.',
 'Python is used in automation.',
 'LangChain is used to build LLM applications.',
 'Chunking divides long text into smaller pieces.']

### Try splitting using a space

In [94]:
splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=50,
    chunk_overlap=0
)

chunks = splitter.split_text(text)
chunks

['Python is easy to learn.\nPython is used in',
 'automation.\n\nLangChain is used to build LLM',
 'applications.\nChunking divides long text into',
 'smaller pieces.']

### Try splitting character by character

In [95]:
splitter = CharacterTextSplitter(
    separator="",
    chunk_size=20,
    chunk_overlap=0
)

chunks = splitter.split_text(text)
chunks

['Python is easy to le',
 'arn.\nPython is used',
 'in automation.\n\nLang',
 'Chain is used to bui',
 'ld LLM applications.',
 'Chunking divides lo',
 'ng text into smaller',
 'pieces.']

## 4. RecursiveCharacterTextSplitter

### What is RecursiveCharacterTextSplitter?

Recursive means step-by-step or in a hierarchy.

It tries different separators in order:
1. Paragraph (`\n\n`)
2. New line (`\n`)
3. Space (` `)
4. Individual character (`""`)

If a paragraph is too large, it tries a smaller boundary. This usually keeps sentences and paragraphs more readable than splitting at only one separator.

### Step 1: Import RecursiveCharacterTextSplitter

In [96]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Step 2: Create sample text with paragraphs

In [108]:
text = """
Paragraph one: Python is a programming language. It is easy to learn.
Python is used in automation and data science.

Paragraph two: LangChain helps developers build applications with language models.
It provides tools to connect models with data sources.
Document splitting divides long text into smaller chunks.

Paragraph three: RAG stands for Retrieval Augmented Generation.
It retrieves relevant information before generating an answer.
"""

print(text)


Paragraph one: Python is a programming language. It is easy to learn.
Python is used in automation and data science.

Paragraph two: LangChain helps developers build applications with language models.
It provides tools to connect models with data sources.
Document splitting divides long text into smaller chunks.

Paragraph three: RAG stands for Retrieval Augmented Generation.
It retrieves relevant information before generating an answer.



### Step 3: Create the recursive splitter

In [115]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=10
)

### Step 4: Split the text

In [116]:
chunks = splitter.split_text(text)

### Step 5: Check the number of chunks

In [117]:
len(chunks)

7

### Step 6: Print each chunk and its character count

In [118]:
for i, chunk in enumerate(chunks):
    print("Chunk:", i + 1)
    print("Characters:", len(chunk))
    print(chunk)
    print("--------------------")

Chunk: 1
Characters: 69
Paragraph one: Python is a programming language. It is easy to learn.
--------------------
Chunk: 2
Characters: 46
Python is used in automation and data science.
--------------------
Chunk: 3
Characters: 82
Paragraph two: LangChain helps developers build applications with language models.
--------------------
Chunk: 4
Characters: 54
It provides tools to connect models with data sources.
--------------------
Chunk: 5
Characters: 57
Document splitting divides long text into smaller chunks.
--------------------
Chunk: 6
Characters: 63
Paragraph three: RAG stands for Retrieval Augmented Generation.
--------------------
Chunk: 7
Characters: 62
It retrieves relevant information before generating an answer.
--------------------


### Understand the separators
You can provide the separators explicitly. The splitter tries them in the given order.

In [102]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=100,
    chunk_overlap=10
)

chunks = splitter.split_text(text)
chunks

['Paragraph one: Python is a programming language. It is easy to learn.',
 'Python is used in automation and data science.',
 'Paragraph two: LangChain helps developers build applications with language models.',
 'It provides tools to connect models with data sources.',
 'Document splitting divides long text into smaller chunks.',
 'Paragraph three: RAG stands for Retrieval Augmented Generation.',
 'It retrieves relevant information before generating an answer.']

## 5. TokenTextSplitter

### What is a token?

A token is a piece of text that a language model processes. A token may be:
- A whole word
- Part of a word
- Punctuation
- A space combined with text

**Character count and token count are different.** LLM usage and context limits are often measured in tokens.

### Step 1: Install and import tiktoken

In [103]:
import tiktoken

### Step 2: Select a tokenizer

In [104]:
encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")

### Step 3: Convert text into token IDs

In [105]:
text = "I love learning LangChain and AI!"
tokens = encoding.encode(text)

print(tokens)

[40, 3021, 6975, 23272, 19368, 323, 15592, 0]


### Step 4: Count tokens

In [106]:
len(tokens)

8

### Step 5: Decode token IDs back to text

In [107]:
encoding.decode(tokens)

'I love learning LangChain and AI!'

### Step 6: Compare characters, words, and tokens

In [32]:
print("Characters:", len(text))
print("Words:", len(text.split()))
print("Tokens:", len(encoding.encode(text)))

Characters: 33
Words: 6
Tokens: 8


### Step 7: Import TokenTextSplitter

In [33]:
from langchain_text_splitters import TokenTextSplitter

### Step 8: Create a token splitter

In [34]:
splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=10,
    chunk_overlap=2
)

### Step 9: Split the text

In [35]:
text = """LangChain helps developers build applications with language models.
Document loading reads information from files.
Chunking divides long documents into smaller pieces.
Token limits are important when working with language models."""

chunks = splitter.split_text(text)
chunks

['LangChain helps developers build applications with language models.\n',
 ' models.\nDocument loading reads information from files.\nChunk',
 '.\nChunking divides long documents into smaller pieces.\n',
 ' pieces.\nToken limits are important when working with language',
 ' with language models.']

### Step 10: Check token count in every chunk

In [36]:
for i, chunk in enumerate(chunks):
    token_count = len(encoding.encode(chunk))
    print("Chunk:", i + 1)
    print("Token count:", token_count)
    print(chunk)
    print("--------------------")

Chunk: 1
Token count: 10
LangChain helps developers build applications with language models.

--------------------
Chunk: 2
Token count: 10
 models.
Document loading reads information from files.
Chunk
--------------------
Chunk: 3
Token count: 10
.
Chunking divides long documents into smaller pieces.

--------------------
Chunk: 4
Token count: 10
 pieces.
Token limits are important when working with language
--------------------
Chunk: 5
Token count: 4
 with language models.
--------------------


**Remember:**
- `CharacterTextSplitter` uses one separator.
- `RecursiveCharacterTextSplitter` tries multiple separators.
- `TokenTextSplitter` splits according to token limits.

## 6. MarkdownHeaderTextSplitter

### What is Markdown?

Markdown is a simple way to format text using headings:
- `#` – Heading 1
- `##` – Heading 2
- `###` – Heading 3

`MarkdownHeaderTextSplitter` splits Markdown content based on headings and stores heading information in metadata.

### Step 1: Import MarkdownHeaderTextSplitter

In [37]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

### Step 2: Create Markdown text

In [38]:
markdown_text = """
# Python

Python is a programming language.

## Variables

Variables store values.

## Functions

Functions are reusable blocks of code.

# LangChain

LangChain helps build applications with language models.

## Chunking

Chunking divides long text into smaller pieces.
"""

print(markdown_text)


# Python

Python is a programming language.

## Variables

Variables store values.

## Functions

Functions are reusable blocks of code.

# LangChain

LangChain helps build applications with language models.

## Chunking

Chunking divides long text into smaller pieces.



### Step 3: Define the Markdown headers

In [39]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2")
]

### Step 4: Create the splitter

In [40]:
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

### Step 5: Split the Markdown text

In [41]:
markdown_documents = markdown_splitter.split_text(markdown_text)

### Step 6: Check number of Documents

In [42]:
len(markdown_documents)

5

### Step 7: Print page content and metadata

In [43]:
for document in markdown_documents:
    print("Content:")
    print(document.page_content)
    print("Metadata:")
    print(document.metadata)
    print("--------------------")

Content:
Python is a programming language.
Metadata:
{'Header 1': 'Python'}
--------------------
Content:
Variables store values.
Metadata:
{'Header 1': 'Python', 'Header 2': 'Variables'}
--------------------
Content:
Functions are reusable blocks of code.
Metadata:
{'Header 1': 'Python', 'Header 2': 'Functions'}
--------------------
Content:
LangChain helps build applications with language models.
Metadata:
{'Header 1': 'LangChain'}
--------------------
Content:
Chunking divides long text into smaller pieces.
Metadata:
{'Header 1': 'LangChain', 'Header 2': 'Chunking'}
--------------------


**Important:** Markdown header splitting uses the heading structure. It does not automatically guarantee that every resulting section is below a chosen character limit. You can apply another splitter afterward if needed.

## 7. HTML Header Text Splitter

### What is HTML?

HTML is used to structure web pages. Common heading tags are:
- `<h1>` – Main heading
- `<h2>` – Subheading
- `<h3>` – Smaller heading

`HTMLHeaderTextSplitter` reads HTML and uses heading tags to organize the extracted text. It can store heading values in metadata.

### Step 1: Import HTMLHeaderTextSplitter

In [44]:
from langchain_text_splitters import HTMLHeaderTextSplitter

### Step 2: Create sample HTML

In [45]:
html_text = """
<html>
<body>
<h1>Python</h1>
<p>Python is a programming language.</p>
<h2>Variables</h2>
<p>Variables store values.</p>
<h2>Functions</h2>
<p>Functions are reusable blocks of code.</p>
<h1>LangChain</h1>
<p>LangChain helps build LLM applications.</p>
<h2>Chunking</h2>
<p>Chunking divides long text into smaller pieces.</p>
</body>
</html>
"""

print(html_text)


<html>
<body>
<h1>Python</h1>
<p>Python is a programming language.</p>
<h2>Variables</h2>
<p>Variables store values.</p>
<h2>Functions</h2>
<p>Functions are reusable blocks of code.</p>
<h1>LangChain</h1>
<p>LangChain helps build LLM applications.</p>
<h2>Chunking</h2>
<p>Chunking divides long text into smaller pieces.</p>
</body>
</html>



### Step 3: Define HTML heading tags

In [46]:
headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2")
]

### Step 4: Create HTML splitter

In [47]:
html_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

### Step 5: Split HTML content

In [48]:
html_documents = html_splitter.split_text(html_text)

### Step 6: Check number of Documents

In [49]:
len(html_documents)

10

### Step 7: Print content and metadata

In [50]:
for document in html_documents:
    print("Content:")
    print(document.page_content)
    print("Metadata:")
    print(document.metadata)
    print("--------------------")

Content:
Python
Metadata:
{'Header 1': 'Python'}
--------------------
Content:
Python is a programming language.
Metadata:
{'Header 1': 'Python'}
--------------------
Content:
Variables
Metadata:
{'Header 1': 'Python', 'Header 2': 'Variables'}
--------------------
Content:
Variables store values.
Metadata:
{'Header 1': 'Python', 'Header 2': 'Variables'}
--------------------
Content:
Functions
Metadata:
{'Header 1': 'Python', 'Header 2': 'Functions'}
--------------------
Content:
Functions are reusable blocks of code.
Metadata:
{'Header 1': 'Python', 'Header 2': 'Functions'}
--------------------
Content:
LangChain
Metadata:
{'Header 1': 'LangChain'}
--------------------
Content:
LangChain helps build LLM applications.
Metadata:
{'Header 1': 'LangChain'}
--------------------
Content:
Chunking
Metadata:
{'Header 1': 'LangChain', 'Header 2': 'Chunking'}
--------------------
Content:
Chunking divides long text into smaller pieces.
Metadata:
{'Header 1': 'LangChain', 'Header 2': 'Chunking'}


### Optional: Load HTML from a local file

In [119]:
from pathlib import Path

html_path = Path("Data_files/sample.html")
html_from_file = html_path.read_text(encoding="utf-8")

html_documents_from_file = html_splitter.split_text(html_from_file)
len(html_documents_from_file)

10

## 8. Chunk Size

### What is Chunk Size?

Chunk size controls the maximum size that a text splitter tries to create for each chunk.

For character-based splitters, chunk size is measured in **characters** when using the default length function.

Example:
- `chunk_size=100` means the splitter tries to create chunks around 100 characters or smaller.
- It is not the same as 100 words or 100 tokens.

### Example: Small chunk size

In [52]:
text = "Python is easy to learn. LangChain helps build LLM applications. Chunking splits long documents."

In [53]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=30,
    chunk_overlap=0
)

small_chunks = splitter.split_text(text)
small_chunks

['Python is easy to learn.',
 'LangChain helps build LLM',
 'applications. Chunking splits',
 'long documents.']

### Example: Larger chunk size

In [54]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=0
)

large_chunks = splitter.split_text(text)
large_chunks

['Python is easy to learn. LangChain helps build LLM applications. Chunking splits',
 'long documents.']

### Compare the number of chunks

In [55]:
print("Small chunk size - number of chunks:", len(small_chunks))
print("Large chunk size - number of chunks:", len(large_chunks))

Small chunk size - number of chunks: 4
Large chunk size - number of chunks: 2


### Print character count for every chunk

In [56]:
for chunk in small_chunks:
    print(len(chunk), "characters:", chunk)

24 characters: Python is easy to learn.
25 characters: LangChain helps build LLM
29 characters: applications. Chunking splits
15 characters: long documents.


**Remember:** Smaller chunks usually create more chunks. Larger chunks usually create fewer chunks. The best size depends on the document, retrieval needs, and model.

## 9. Chunk Overlap

### What is Chunk Overlap?

Chunk overlap means repeating a small part of one chunk at the beginning of the next chunk.

Why?
- A sentence or idea may be split between two chunks.
- Overlap keeps some neighboring context.
- It can help retrieval preserve the connection between chunks.

Example:
- Chunk size = 50 characters
- Chunk overlap = 10 characters
- Around 10 characters from the previous chunk can be repeated in the next one.

### Step 1: Create sample text

In [57]:
text = """
Python is a programming language.
It is used for automation and data science.
LangChain helps developers build LLM applications.
Chunking divides long documents into smaller pieces.
"""

### Step 2: Split without overlap

In [58]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=0
)

chunks_without_overlap = splitter.split_text(text)
chunks_without_overlap

['Python is a programming language.',
 'It is used for automation and data science.',
 'LangChain helps developers build LLM',
 'applications.',
 'Chunking divides long documents into smaller',
 'pieces.']

### Step 3: Split with overlap

In [59]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

chunks_with_overlap = splitter.split_text(text)
chunks_with_overlap

['Python is a programming language.',
 'It is used for automation and data science.',
 'LangChain helps developers build LLM',
 'build LLM applications.',
 'Chunking divides long documents into smaller',
 'smaller pieces.']

### Step 4: Compare both results

In [60]:
print("Without overlap:")
for chunk in chunks_without_overlap:
    print(chunk)
    print("----")

print("\nWith overlap:")
for chunk in chunks_with_overlap:
    print(chunk)
    print("----")

Without overlap:
Python is a programming language.
----
It is used for automation and data science.
----
LangChain helps developers build LLM
----
applications.
----
Chunking divides long documents into smaller
----
pieces.
----

With overlap:
Python is a programming language.
----
It is used for automation and data science.
----
LangChain helps developers build LLM
----
build LLM applications.
----
Chunking divides long documents into smaller
----
smaller pieces.
----


**Important:** `chunk_overlap` must be smaller than `chunk_size`. Overlap is measured using the splitter's length unit (characters by default for this splitter).

## 10. Semantic Chunking

### What is Semantic Chunking?

Semantic means **related to meaning**.

Normal splitters divide text based on characters, separators, or tokens. Semantic chunking tries to keep sentences with related meanings together.

Simple idea:
1. Divide the text into sentences.
2. Convert sentences into embeddings.
3. Compare the meaning of neighboring sentences.
4. Split when the meaning changes enough.

Semantic chunking needs an embedding model. This example uses a local Sentence Transformers model. The first run may download the model and needs internet access.

### Step 1: Import SentenceTransformerEmbeddings

In [61]:
from langchain_huggingface import HuggingFaceEmbeddings

### Step 2: Create an embedding model

In [62]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\vvard\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4477.84it/s]


### Step 3: Import SemanticChunker

In [63]:
from langchain_experimental.text_splitter import SemanticChunker

C:\Users\vvard\AppData\Local\Temp\ipykernel_1304\2829801429.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


### Step 4: Create SemanticChunker

In [64]:
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

### Step 5: Create example text with different topics

In [65]:
semantic_text = """
Python is a programming language.
Python supports variables, loops, and functions.
Developers use Python for automation and data analysis.

The Amazon rainforest is home to many plants and animals.
It has a warm climate and receives heavy rainfall.
Many species live in this forest ecosystem.

LangChain helps developers build applications with language models.
Document loaders read files and text splitters divide documents into chunks.
"""

print(semantic_text)


Python is a programming language.
Python supports variables, loops, and functions.
Developers use Python for automation and data analysis.

The Amazon rainforest is home to many plants and animals.
It has a warm climate and receives heavy rainfall.
Many species live in this forest ecosystem.

LangChain helps developers build applications with language models.
Document loaders read files and text splitters divide documents into chunks.



### Step 6: Split the text based on meaning

In [66]:
semantic_chunks = semantic_splitter.split_text(semantic_text)

### Step 7: Print semantic chunks

In [67]:
for i, chunk in enumerate(semantic_chunks):
    print("Chunk:", i + 1)
    print(chunk)
    print("--------------------")

Chunk: 1

Python is a programming language. Python supports variables, loops, and functions. Developers use Python for automation and data analysis. The Amazon rainforest is home to many plants and animals. It has a warm climate and receives heavy rainfall.
--------------------
Chunk: 2
Many species live in this forest ecosystem. LangChain helps developers build applications with language models. Document loaders read files and text splitters divide documents into chunks. 
--------------------


**Remember:**
- Semantic chunking requires embeddings.
- It tries to keep related ideas together.
- Chunk sizes may be different.
- It does not use the same fixed `chunk_size` and `chunk_overlap` controls as a recursive splitter.

## 11. Parent-Child Chunking

### What is Parent-Child Chunking?

Parent-child chunking creates two levels of chunks:

- **Parent chunk:** A larger section that keeps more context.
- **Child chunk:** A smaller piece inside the parent. It can be used for more precise searching.

Simple example:
- Parent: one large paragraph about Python.
- Children: smaller sentences or sections from that paragraph.

When a child chunk matches a search, an application can return its parent chunk to provide more context.

### Step 1: Import RecursiveCharacterTextSplitter

In [68]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Step 2: Create a sample document

In [69]:
parent_text = """
Python is a programming language. It is easy to learn.
Python is used in automation, web development, and data science.
Python supports functions, classes, and modules.

LangChain is a framework for building applications with language models.
It connects language models with data sources and tools.
LangChain can be used to build retrieval augmented generation applications.

Chunking divides long documents into smaller pieces.
Parent-child chunking creates large parent chunks and smaller child chunks.
A matching child can point back to its parent for additional context.
"""

print(parent_text)


Python is a programming language. It is easy to learn.
Python is used in automation, web development, and data science.
Python supports functions, classes, and modules.

LangChain is a framework for building applications with language models.
It connects language models with data sources and tools.
LangChain can be used to build retrieval augmented generation applications.

Chunking divides long documents into smaller pieces.
Parent-child chunking creates large parent chunks and smaller child chunks.
A matching child can point back to its parent for additional context.



### Step 3: Create the parent splitter
Parent chunks are larger sections.

In [70]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

### Step 4: Create the child splitter
Child chunks are smaller sections.

In [71]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=10
)

### Step 5: Create parent chunks

In [72]:
parent_chunks = parent_splitter.split_text(parent_text)

print("Number of parent chunks:", len(parent_chunks))

Number of parent chunks: 4


### Step 6: Print parent chunks

In [73]:
for i, parent in enumerate(parent_chunks):
    print("Parent:", i + 1)
    print(parent)
    print("--------------------")

Parent: 1
Python is a programming language. It is easy to learn.
Python is used in automation, web development, and data science.
Python supports functions, classes, and modules.
--------------------
Parent: 2
LangChain is a framework for building applications with language models.
It connects language models with data sources and tools.
--------------------
Parent: 3
LangChain can be used to build retrieval augmented generation applications.
--------------------
Parent: 4
Chunking divides long documents into smaller pieces.
Parent-child chunking creates large parent chunks and smaller child chunks.
A matching child can point back to its parent for additional context.
--------------------


### Step 7: Split each parent into child chunks
Each child is stored with the number of its parent. This is a simple demonstration of the parent-child relationship.

In [74]:
child_records = []

for parent_id, parent_text in enumerate(parent_chunks):
    child_chunks = child_splitter.split_text(parent_text)

    for child_text in child_chunks:
        child_records.append({
            "parent_id": parent_id,
            "child_text": child_text
        })

print("Number of child chunks:", len(child_records))

Number of child chunks: 9


### Step 8: Print child chunks and their parent IDs

In [75]:
for child in child_records:
    print("Parent ID:", child["parent_id"])
    print("Child text:", child["child_text"])
    print("--------------------")

Parent ID: 0
Child text: Python is a programming language. It is easy to learn.
--------------------
Parent ID: 0
Child text: Python is used in automation, web development, and data science.
--------------------
Parent ID: 0
Child text: Python supports functions, classes, and modules.
--------------------
Parent ID: 1
Child text: LangChain is a framework for building applications with language models.
--------------------
Parent ID: 1
Child text: It connects language models with data sources and tools.
--------------------
Parent ID: 2
Child text: LangChain can be used to build retrieval augmented generation applications.
--------------------
Parent ID: 3
Child text: Chunking divides long documents into smaller pieces.
--------------------
Parent ID: 3
Child text: Parent-child chunking creates large parent chunks and smaller child chunks.
--------------------
Parent ID: 3
Child text: A matching child can point back to its parent for additional context.
--------------------


### Step 9: Find the parent for a selected child
In a real RAG application, the child is retrieved by similarity search and its parent ID is used to fetch the larger parent text.

In [76]:
selected_child = child_records[0]
selected_parent_id = selected_child["parent_id"]

print("Selected child:")
print(selected_child["child_text"])

print("\nRelated parent:")
print(parent_chunks[selected_parent_id])

Selected child:
Python is a programming language. It is easy to learn.

Related parent:
Python is a programming language. It is easy to learn.
Python is used in automation, web development, and data science.
Python supports functions, classes, and modules.


**Important:** This section demonstrates the parent-child chunk relationship using simple Python lists and dictionaries. A production RAG application can store child chunks in a vector database and keep parent documents in a document store, using LangChain's `ParentDocumentRetriever`.

## Final Comparison

| Splitter | How does it split? | Needs an embedding model? |
|---|---|---|
| CharacterTextSplitter | One separator | No |
| RecursiveCharacterTextSplitter | Tries multiple separators | No |
| TokenTextSplitter | Token count | No separate embedding model |
| MarkdownHeaderTextSplitter | Markdown headings | No |
| HTMLHeaderTextSplitter | HTML heading tags | No |
| SemanticChunker | Meaning / semantic similarity | Yes |
| Parent-child approach | Larger parent + smaller child pieces | Not for splitting itself |

## Practice Tasks

1. Change `chunk_size` from 50 to 100. Compare the number of chunks.
2. Change `chunk_overlap` from 0 to 10. Observe repeated text.
3. Try `separator=" "` with CharacterTextSplitter.
4. Add another Markdown heading and run MarkdownHeaderTextSplitter.
5. Add an `<h2>` tag to the HTML example.
6. Try your own paragraph with TokenTextSplitter.
7. Change the semantic text to include two unrelated topics.
8. Increase and decrease parent and child chunk sizes.

## Final RAG Flow

`Load Documents → Split into Chunks → Create Embeddings → Store in Vector Database → Retrieve Relevant Chunks → Send Context to LLM`